In [1]:
%pip install -q pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib

import pandas as pd
import src.constants

importlib.reload(src.constants)
from src.constants import (
    Column,
    MODEL_FEATURE_COLUMNS,
    OUTPUT_DIR,
)

In [3]:


def calculate_ratios(df: pd.DataFrame):     
    df[Column.LABEL] = df[Column.LABEL].map({"OR": 0, "CG": 1})

    df[Column.ADJECTIVE_RATIO] = df[Column.ADJECTIVE_COUNT] / df[Column.WORD_COUNT]
    df[Column.VERB_RATIO] = df[Column.VERB_COUNT] / df[Column.WORD_COUNT]
    df[Column.SUPERLATIVE_RATIO] = df[Column.SUPERLATIVE_COUNT] / df[Column.WORD_COUNT]

    df[Column.FIRST_PERSON_PRONOUN_RATIO] = (
        df[Column.FIRST_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )
    df[Column.THIRD_PERSON_PRONOUN_RATIO] = (
        df[Column.THIRD_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )

    claim_count = (
        df[Column.SUBJECTIVE_CLAIM_COUNT]
        + df[Column.OBJECTIVE_CLAIM_COUNT]
    )
    df[Column.SUBJECTIVITY] = (
        df[Column.SUBJECTIVE_CLAIM_COUNT] / claim_count.mask(claim_count == 0)
    ).fillna(0.5)

    normalized_claim_count = claim_count.mask(claim_count == 0)
    df[Column.EXPERIENTIAL_DETAIL] = (
        df[Column.EXPERIENTIAL_DETAIL_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.POSITIVE_AFFECT] = (
        df[Column.POSITIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.NEGATIVE_AFFECT] = (
        df[Column.NEGATIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.UNCERTAINTY] = (
        df[Column.UNCERTAIN_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)

    text_sentiment_normalized = df[Column.TEXT_SENTIMENT] / 2
    rating_sentiment = (df[Column.RATING] - 3) / 2
    df[Column.INTERNAL_CONSISTENCY] = (
        1 - (text_sentiment_normalized - rating_sentiment).abs() / 2
    ).clip(0, 1)

    print(df.columns)
    print(df.head())

def merge(deterministic_dir, output_llm_dir, output):
    deterministics_df = pd.read_csv(deterministic_dir)
    output_llm = pd.read_csv(output_llm_dir)

    final = pd.merge(deterministics_df, output_llm, "inner", on="id")

    calculate_ratios(final)

    final_columns = [Column.ID, Column.LABEL, *MODEL_FEATURE_COLUMNS]
    final = final[final_columns].copy()
    final.to_csv(output, index=None)

In [ ]:
deterministic_dir = OUTPUT_DIR / "sample_deterministic_features_results.csv"
output_llm_dir = OUTPUT_DIR / "llm" / "qwen3_14b_results.csv"
output_dir = OUTPUT_DIR / "final_qwen.csv"